In [1]:
import asyncio
import logging
from question_prompt import PromptGenerator
from question_generate import AsyncQuestionGenerator
from question_quality_checker import QuestionQualityChecker
from question_mapper import QuestionMapper
from config import (
    BANKING_PERSONA_KPI,
    BANKING_QUESTIONS_PERSONA_KPI_MAPPING,
    BANKING_QUESTIONS_PROMPTS,
    BANKING_SYNTHETIC_QUESTIONS,
    BANKING_MAPPED_QUESTIONS,
    ensure_directories_exist,
    QUALITY_CHECK_REPORT_MAPPED_QUESTIONS,
    QUALITY_CHECK_REPORT_SYNTHETIC_QUESTIONS
)
from dotenv import load_dotenv
import os
load_dotenv()
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)
ensure_directories_exist()

In [8]:
# # Option 1: Use config paths (recommended)
# generator = PromptGenerator(
#     persona_kpi_path=str(BANKING_PERSONA_KPI),
#     golden_data_path=str(BANKING_QUESTIONS_PERSONA_KPI_MAPPING),
#     output_path="test_p.csv",   )

# df_prompts = generator.generate_prompts()
# print(f"Generated {len(df_prompts)} prompts")
# df_prompts.head()


2025-12-31 21:00:51,106 - INFO - Loading data mapping and golden examples from: /Users/sulbhajain/Documents/Personal/tursio/data_bank/banking_persona_kpi.csv and /Users/sulbhajain/Documents/Personal/tursio/data_bank/banking_questions_persona_kpi_mapping.csv
2025-12-31 21:00:51,112 - INFO - Data mapping shape: (14, 2), Data golden shape: (14, 4)
2025-12-31 21:00:51,130 - INFO - Prompts saved to test_p.csv of size (42, 5)


Generated 42 prompts


,id,persona,kpi,difficulty,prompt
0,0,Risk & Credit Analytics Manager; CRO; Complian...,90+ DPD Rate; Delinquency Ratio; Non-Performin...,simple,You are a Risk & Credit Analytics Manager; CRO...
1,1,Finance / Treasury Manager; Branch / Regional ...,Total Deposit Balance by Geography; Concentrat...,simple,You are a Finance / Treasury Manager; Branch /...
2,2,Product Manager; Member Analytics Lead,Early Closure Rate; Account Churn (≤90 days),simple,You are a Product Manager; Member Analytics Le...
3,3,Finance Manager; CRO; Product Manager,Portfolio Mix %; Credit Card Exposure by Balance,simple,You are a Finance Manager; CRO; Product Manage...
4,4,Member Analytics Lead; Compliance Officer,Dormant Account Rate; Inactivity Ratio,simple,You are a Member Analytics Lead; Compliance Of...


In [17]:
# question_generator = AsyncQuestionGenerator(
#     max_concurrent=10,
#     output_csv="test_p1.csv",
#     model=os.getenv("OPENAI_41_MODEL"),
#     input_csv="test_p.csv",
# )
# results = await question_generator.run_async()

2025-12-31 21:14:08,749 - INFO - Starting async question generation pipeline
2025-12-31 21:14:08,749 - INFO - Loading data from test_p.csv
2025-12-31 21:14:08,751 - INFO - Loaded 42 rows
2025-12-31 21:14:08,752 - INFO - Generating responses for 42 rows with max 10 concurrent requests
2025-12-31 21:14:08,753 - INFO - Processing row 0
2025-12-31 21:14:08,754 - INFO - Processing row 1
2025-12-31 21:14:08,755 - INFO - Processing row 2
2025-12-31 21:14:08,756 - INFO - Processing row 3
2025-12-31 21:14:08,756 - INFO - Processing row 4
2025-12-31 21:14:08,757 - INFO - Processing row 5
2025-12-31 21:14:08,758 - INFO - Processing row 6
2025-12-31 21:14:08,758 - INFO - Processing row 7
2025-12-31 21:14:08,759 - INFO - Processing row 8
2025-12-31 21:14:08,760 - INFO - Processing row 9
2025-12-31 21:14:10,725 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-12-31 21:14:10,728 - INFO - Response received for row 2
2025-12-31 21:14:10,728 - INFO - Processi

In [2]:

qqc = QuestionQualityChecker(
            input_csv="test_p1.csv",
            question_column="synthetic_questions",
            output_path=str(QUALITY_CHECK_REPORT_SYNTHETIC_QUESTIONS)
        )
report = qqc.run_all_checks()
logger.info(f"Quality Check Report: {len(report)} checks completed.")


2025-12-31 22:38:45,298 - INFO - Found 0 duplicate questions.
2025-12-31 22:38:45,320 - INFO - Found 5 similar question pairs with similarity >= 0.8.
2025-12-31 22:38:45,321 - INFO - Found 0 questions outside the length range of 5-50 words.
2025-12-31 22:38:45,322 - INFO - Found 0 questions with type-token ratio below 0.5.
2025-12-31 22:38:45,324 - INFO - Found 18 questions with stopword ratio above 0.5.
2025-12-31 22:38:45,325 - INFO - Quality Check Summary:
2025-12-31 22:38:45,325 - INFO -   duplicates: 0 issues found
2025-12-31 22:38:45,325 - INFO -   similar_questions: 5 issues found
2025-12-31 22:38:45,326 - INFO -   length_outliers: 0 issues found
2025-12-31 22:38:45,326 - INFO -   low_lexical_diversity: 0 issues found
2025-12-31 22:38:45,326 - INFO -   high_stopword_ratio: 18 issues found
2025-12-31 22:38:45,328 - INFO -   similar_questions results saved to /Users/sulbhajain/Documents/Personal/tursio/quality_checks/synthetic_questions_quality_report/similar_questions_report.csv


In [2]:
# Use real Claude model (2025 version)
model_name = os.getenv("ANTHROPIC_CLAUDE_4_5_HAIKU_MODEL")

logger.info(f"Initializing mapper with model: {model_name}")
mapper = QuestionMapper(
    synthetic_questions_path="test_p1.csv",
    output_path="test_p2.csv",
    model=model_name,
    batch_size=5,  # Reduced batch size to avoid rate limits
)

logger.info(f"Starting question mapping with {len(mapper.df)} questions...")
try:
    await mapper.map_and_save()
    print("\n✓ Question mapping completed!")
except Exception as e:
    logger.error(f"Error during mapping: {e}")
    print(f"✗ Error: {e}")


2025-12-31 22:33:57,380 - INFO - Initializing mapper with model: claude-haiku-4-5-20251001
2025-12-31 22:33:57,384 - INFO - Loaded 42 questions
2025-12-31 22:33:57,415 - INFO - Initialized QuestionMapper with model: claude-haiku-4-5-20251001
2025-12-31 22:33:57,415 - INFO - Starting question mapping with 42 questions...
2025-12-31 22:33:57,415 - INFO - Loading synthetic questions from test_p1.csv
2025-12-31 22:33:57,416 - INFO - Processing batch 1/9 (1-5 of 42)
2025-12-31 22:33:58,304 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2025-12-31 22:33:58,339 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2025-12-31 22:33:58,366 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2025-12-31 22:33:58,391 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2025-12-31 22:33:58,425 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 20


✓ Question mapping completed!


In [3]:
qqc = QuestionQualityChecker(
    input_csv="test_p2.csv",
    question_column="mapped_question",
    output_path=str(QUALITY_CHECK_REPORT_MAPPED_QUESTIONS)
)
report = qqc.run_all_checks()
logger.info(f" Quality Check Report: {len(report)} checks completed.")


2025-12-31 22:39:04,267 - INFO - Found 18 duplicate questions.
2025-12-31 22:39:04,271 - INFO - Found 0 similar question pairs with similarity >= 0.8.
2025-12-31 22:39:04,272 - INFO - Found 18 questions outside the length range of 5-50 words.
2025-12-31 22:39:04,274 - INFO - Found 0 questions with type-token ratio below 0.5.
2025-12-31 22:39:04,275 - INFO - Found 0 questions with stopword ratio above 0.5.
2025-12-31 22:39:04,276 - INFO - Quality Check Summary:
2025-12-31 22:39:04,276 - INFO -   duplicates: 18 issues found
2025-12-31 22:39:04,276 - INFO -   similar_questions: 0 issues found
2025-12-31 22:39:04,276 - INFO -   length_outliers: 18 issues found
2025-12-31 22:39:04,276 - INFO -   low_lexical_diversity: 0 issues found
2025-12-31 22:39:04,277 - INFO -   high_stopword_ratio: 0 issues found
2025-12-31 22:39:04,278 - INFO -   duplicates results saved to /Users/sulbhajain/Documents/Personal/tursio/quality_checks/mapped_questions_quality_report/duplicates_report.csv
2025-12-31 22:3

In [2]:
from response_eval import ResponseEvaluator
reevaluator = ResponseEvaluator(
    data_dir="benchmark_gpt",
    output_dir="test_gpt"
)   
report = reevaluator.run_all_evaluations()
logger.info(f" Response Evaluation Report: {len(report)} responses evaluated.")

/Users/sulbhajain/Documents/Personal/tursio/tursio_env/lib/python3.14/site-packages/posthog/consumer.py:92: SyntaxWarning: 'return' in a 'finally' block
  return success
/Users/sulbhajain/Documents/Personal/tursio/tursio_env/lib/python3.14/site-packages/deepeval/evaluate/execute.py:2744: SyntaxWarning: 'return' in a 'finally' block
  return
2025-12-31 22:51:45,080 - INFO - Running evaluations for simple difficulty
2025-12-31 22:51:45,084 - INFO - Loaded 60 benchmark records for simple difficulty
2025-12-31 22:51:45,098 - INFO - Created dataset with 60 test cases
2025-12-31 22:51:45,098 - INFO - Evaluating test cases 0 to 2


✨ You're running DeepEval's latest Answer Relevancy Metric! (using gpt-4.1, strict=False, async_mode=False)...

✨ You're running DeepEval's latest Bias Metric! (using gpt-4.1, strict=False, async_mode=False)...

Output()

2025-12-31 22:51:46,747 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


...



Metrics Summary

  - ✅ Focus [Conversational GEval] (score: 1.0, threshold: 0.5, strict: False, evaluation model: gpt-4.1, reason: The assistant directly addresses the user's question about customer service interaction and account retention, providing a relevant, complete, and coherent response. The content is appropriate for the assistant's role and maintains a clear connection to the user's query, fully meeting the evaluation steps., error: None)
  - ✅ Helpful [Conversational GEval] (score: 1.0, threshold: 0.5, strict: False, evaluation model: gpt-4.1, reason: The assistant directly addresses the user's question about customer service interaction and account retention, providing a clear, accurate, and meaningful explanation. The response covers key factors such as issue resolution, proactive outreach, and the importance of interaction quality, fully advancing the user's goal., error: None)
  - ✅ Voice [Conversational GEval] (score: 0.9989013057642644, threshold: 0.5, strict: False,

⚠ WARNING: No hyperparameters logged.
» ]8;id=347820;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 15.36s | token cost: 0.017703999999999998 USD)
» Test Results (2 total tests):
   » Pass Rate: 100.0% | Passed: 2 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

2025-12-31 23:44:46,533 - INFO - Evaluation completed in 399.65 seconds
2025-12-31 23:44:46,547 - INFO - Results saved to test_gpt/gpt_hard_convo/evaluation_results.csv


TypeError: object of type 'NoneType' has no len()

Testing actual Claude model availability...



2025-12-31 22:00:08,060 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


✓ claude-sonnet-4-5-20250929 - AVAILABLE

✓ Working model found: claude-sonnet-4-5-20250929
Update ANTHROPIC_CLAUDE_3_5_SONNET_MODEL in your .env file to: claude-sonnet-4-5-20250929
